# R5: December 2015 Detections, Manual Review, and Population Metrics

The temporal-generalization test of the paper: the model, trained on
Dec 2013 to Jan 2015 data, is applied unchanged to all of December 2015,
over ten months after its latest training image, across substantial changes
in lake level, crater geometry, and plume behavior.


In [ ]:
# --- Repository setup -------------------------------------------------------
# Locates config.py at the repository root so this notebook runs no matter
# which directory Jupyter was launched from.  All paths and constants live in
# config.py; edit that one file rather than anything below.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config
config.describe()


In [ ]:
import os, pickle, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

CSV_PATH = config.CSV_2015_12          # override here to analyze another month
random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)


## 1. Load, filter, smooth, renormalize


In [ ]:
df = pd.read_csv(CSV_PATH, header=None,
                 names=["Timestamp_Str", "Large Eruption", "No Eruption", "Small Eruption"])
df["Timestamp"] = pd.to_datetime(df["Timestamp_Str"], format="%Y-%m-%d_%H%M%S_%f")
print(f"Raw rows loaded:   {len(df):,}")

# Isolation filter: drop frames with no neighbor within 10 s on both sides
df = df.sort_values("Timestamp").reset_index(drop=True)
dt_prev = df["Timestamp"].diff()
dt_next = df["Timestamp"].diff(periods=-1).abs()
isolated = (((dt_prev > pd.Timedelta(seconds=10)) | dt_prev.isna()) &
            ((dt_next > pd.Timedelta(seconds=10)) | dt_next.isna()))
df = df[~isolated].reset_index(drop=True)
print(f"Isolated removed:  {int(isolated.sum()):,}")

large_s = pd.concat([df["Large Eruption"].shift(-i) for i in range(config.SMOOTH_LARGE)], axis=1).mean(axis=1)
small_s = pd.concat([df["Small Eruption"].shift(-i) for i in range(config.SMOOTH_OTHER)], axis=1).mean(axis=1)
no_s    = pd.concat([df["No Eruption"].shift(-i)    for i in range(config.SMOOTH_OTHER)], axis=1).mean(axis=1)
df["Large Eruption"], df["Small Eruption"], df["No Eruption"] = large_s, small_s, no_s

prob_cols = ["Large Eruption", "Small Eruption", "No Eruption"]
df[prob_cols] = df[prob_cols].div(df[prob_cols].sum(axis=1), axis=0)
before = len(df)
df = df.dropna(subset=prob_cols).reset_index(drop=True)
df["csv_index"] = df.index
df["delta"] = df["Timestamp"].diff()
print(f"Smoothing tail dropped: {before - len(df):,}")
print(f"Final usable rows: {len(df):,}")


## 2. Dual-threshold classification and event clustering

Expected: **49 Large** events and **420 Small** events.

In [ ]:
pl, ps, pn = (df[c].to_numpy() for c in ["Large Eruption", "Small Eruption", "No Eruption"])
m = np.maximum.reduce([pl, ps, pn])
pred = np.full(len(df), "No Eruption", dtype=object)
pred[(m == pl) & (m != pn) & (pl > config.THRESH_LARGE)] = "Large Eruption"
pred[(m == ps) & (m != pn) & (m != pl) & (ps > config.THRESH_SMALL)] = "Small Eruption"
df["Pred_Label"] = pred
print("Frame-level predictions:")
print(df["Pred_Label"].value_counts().to_string())

def cluster_into_events(sub, gap_sec):
    """Group consecutive rows whose gap <= gap_sec; returns start/end times and CSV index range."""
    if sub.empty:
        return pd.DataFrame(columns=["Start", "End", "Label", "csv_start", "csv_end"])
    sub = sub.sort_values("Timestamp").reset_index(drop=True)
    events, s = [], 0
    for i in range(1, len(sub)):
        if (sub["Timestamp"][i] - sub["Timestamp"][i - 1]).total_seconds() > gap_sec:
            events.append((s, i - 1)); s = i
    events.append((s, len(sub) - 1))
    return pd.DataFrame([dict(Start=sub["Timestamp"][a], End=sub["Timestamp"][b],
                              Label=sub["Pred_Label"][a],
                              csv_start=int(sub["csv_index"][a]), csv_end=int(sub["csv_index"][b]))
                         for a, b in events])

large_events = cluster_into_events(df[df["Pred_Label"] == "Large Eruption"], config.LARGE_EVENT_GAP_SEC)
small_events = cluster_into_events(df[df["Pred_Label"] == "Small Eruption"], config.EVENT_GAP_SEC)
print(f"\nLarge events: {len(large_events)}   (expected 49; gap = {config.LARGE_EVENT_GAP_SEC} s)")
print(f"Small events: {len(small_events)}   (expected 420; gap = {config.EVENT_GAP_SEC} s)")


## 3. Build the review queue

Every Large and Small detection is queued in full (weight 1). Each event
carries a 5-image window for visual review: the first five frames of a Large
event; one frame either side of a Small event's three frames.

In [ ]:
N = len(df)

def window_indices(start_ix, label):
    start = start_ix if label == "Large Eruption" else start_ix - 1
    start = max(0, min(start, N - 5))
    return list(range(start, start + 5))

def rgb_path(ts_str):
    """Path of a frame in the RGB archive: <root>/YYYY/MM/DD/HH/<ts>.jpg (None if archive absent)."""
    try:
        date_part, time_part = ts_str.split("_")[:2]
        y, mth, d = date_part.split("-")
        return str(config.RGB_ARCHIVE_DIR / y / mth / d / time_part[:2] / (ts_str + ".jpg"))
    except Exception:
        return None

review_items = []
for _, ev in pd.concat([large_events, small_events], ignore_index=True).sort_values("Start").iterrows():
    idxs = window_indices(int(ev["csv_start"]), ev["Label"])
    rows = df.iloc[idxs]
    review_items.append(dict(
        category=ev["Label"], center_time=ev["Start"], center_index=int(ev["csv_start"]),
        window_idxs=idxs, image_paths=[rgb_path(t) for t in rows["Timestamp_Str"]],
        max_prob=float(df.iloc[int(ev["csv_start"]):int(ev["csv_end"]) + 1][ev["Label"]].max()),
        weight=1.0, stratum="full_review"))
print(f"Eruption events queued: {len(review_items)}   (expected 469)")


## 4. Stratified No-Eruption sampling (Horvitz-Thompson design)

Reviewing every one of >1.1 M no-eruption frames is impossible, so frames
are binned by their *runner-up* probability (how close the model came to
calling an eruption) and sampled: the near-threshold bin exhaustively, the
rest at 300 each. Each reviewed item is weighted by its bin's *N_h / n_h*,
so weighted counts estimate the whole month without bias; the weights
summing exactly to the candidate pool confirms the design.


In [ ]:
ts_sec = config.seconds_since_epoch(df["Timestamp"])
eruption_ts = np.sort(ts_sec[(df["Pred_Label"] != "No Eruption").to_numpy()])
ins = np.searchsorted(eruption_ts, ts_sec)
left = np.clip(ins - 1, 0, len(eruption_ts) - 1)
right = np.clip(ins, 0, len(eruption_ts) - 1)
dist = np.minimum(np.abs(ts_sec - eruption_ts[left]), np.abs(ts_sec - eruption_ts[right]))
window_min = pd.Series(dist).rolling(5, center=True, min_periods=5).min().to_numpy()

candidate = ((df["Pred_Label"].to_numpy() == "No Eruption")
             & ~np.isnan(window_min) & (window_min >= config.NO_ERUPTION_BUFFER_SEC))
candidate_idxs = np.flatnonzero(candidate)
print(f"No-Eruption candidate pool: {len(candidate_idxs):,}   (expected 1,156,274)")

runner_up = df[["Large Eruption", "Small Eruption"]].max(axis=1).to_numpy()
rng = np.random.default_rng(config.RANDOM_SEED)
total_weight = 0.0
print(f"\n{'stratum':16s} {'range':>14s} {'N_h':>10s} {'n_h':>6s} {'weight':>10s}")
for stratum in config.STRATA:
    in_bin = ((runner_up[candidate_idxs] >= stratum["lo"])
              & (runner_up[candidate_idxs] < stratum["hi"]))
    bin_idxs = candidate_idxs[in_bin]
    N_h = len(bin_idxs)
    if stratum["target"] is None or stratum["target"] >= N_h:
        n_h, chosen = N_h, bin_idxs
    else:
        n_h = stratum["target"]
        chosen = rng.choice(bin_idxs, size=n_h, replace=False)
    weight = N_h / n_h
    total_weight += weight * n_h
    print(f"{stratum['name']:16s} [{stratum['lo']:.2f},{stratum['hi']:.2f}) "
          f"{N_h:>10,} {n_h:>6,} {weight:>10.2f}")
    for idx in chosen:
        idx = int(idx)
        window = list(range(idx - 2, idx + 3))
        review_items.append(dict(
            category="No Eruption", center_time=df.iloc[idx]["Timestamp"], center_index=idx,
            window_idxs=window,
            image_paths=[rgb_path(t) for t in df.iloc[window]["Timestamp_Str"]],
            max_prob=float(df.iloc[idx]["No Eruption"]),
            weight=weight, stratum=stratum["name"]))

print(f"\nSum of weights: {total_weight:,.0f}   (must equal the candidate pool)")
print(f"Total review queue: {len(review_items):,}   (expected 2,930)")
for r in review_items:
    r["id"] = f"{r['category']}|{r['center_index']}"


## 5. Load the manual-review outcomes

`data/review_results_2015_12.csv` records the human verdict for every one of
the 2,930 reviewed events: the model's predicted class, the reviewer's true
class, the stratum, and the Horvitz-Thompson weight. The metrics sections
below are driven entirely by this file, so they reproduce the paper's
Table 2 and F1 values with no images and no clicking.

The validation block cross-checks the file against the pipeline just
rebuilt: the 469 detection events and all 1,561 near-threshold items must
match index-for-index (both are deterministic functions of the CSV), and
every sampled background item must fall inside its stated stratum. To redo
the review from scratch instead, use the interactive appendix (Path B) at
the end of this notebook: its export cell writes a file in this same
format.

In [ ]:
res = None
if config.REVIEW_RESULTS_CSV.exists():
    res = pd.read_csv(config.REVIEW_RESULTS_CSV)
    print(f"Loaded {len(res):,} review outcomes from {config.REVIEW_RESULTS_CSV.name}\n")

    # -- Cross-validate the file against the pipeline rebuilt above --
    his_large = set(res[res["predicted"] == "Large Eruption"]["center_index"])
    his_small = set(res[res["predicted"] == "Small Eruption"]["center_index"])
    mine_large = set(int(i) for i in large_events["csv_start"])
    mine_small = set(int(i) for i in small_events["csv_start"])
    print(f"Large detections match pipeline: {his_large == mine_large} "
          f"({len(his_large)} events)")
    print(f"Small detections match pipeline: {his_small == mine_small} "
          f"({len(his_small)} events)")

    stratum_of = {r["center_index"]: r["stratum"] for r in review_items
                  if r["category"] == "No Eruption"}
    near_mine = {i for i, s in stratum_of.items() if s == "near_threshold"}
    near_his = set(res[res["stratum"] == "near_threshold"]["center_index"])
    print(f"Near-threshold stratum matches:   {near_his == near_mine} "
          f"({len(near_his)} items)")

    pools = {}
    for stratum in config.STRATA:
        in_bin = ((runner_up[candidate_idxs] >= stratum["lo"])
                  & (runner_up[candidate_idxs] < stratum["hi"]))
        pools[stratum["name"]] = set(candidate_idxs[in_bin].tolist())
    for name in ("moderate", "low", "clear"):
        his = set(res[res["stratum"] == name]["center_index"])
        print(f"Sampled '{name}' items all inside their stratum pool: {his <= pools[name]}")

    counts = res["predicted"].value_counts()
    print(f"\nReviewed detections: {counts.get('Large Eruption', 0)} large, "
          f"{counts.get('Small Eruption', 0)} small, "
          f"{counts.get('No Eruption', 0)} background items")
else:
    print("data/review_results_2015_12.csv not found - the metrics sections "
          "below will be skipped.\nRe-create it with the interactive review "
          "(Path B) at the end of this notebook.")


## 6. Confusion matrices and F1: Table 2 and the population-weighted metrics

The raw matrix counts the reviewed items one-for-one; it is Table 2 of the
paper. The weighted matrix multiplies each item by its sampling weight,
turning the review sample into an unbiased estimate of the full month.

Terminology, matching the paper: the per-class precision/recall/F1 are
computed on the weighted matrix (for the detection classes the two
matrices coincide, since every detection has weight 1). The
population-weighted macro F1 (paper: 0.9332) is the mean of the three
weighted per-class F1 scores, and the *eruption-only macro F1 (paper:
0.8998) is the mean of the Large and Small scores, excluding the dominant
background class.

In [ ]:
LABEL_ORDER = ["Large Eruption", "Small Eruption", "No Eruption"]
lab_ix = {l: i for i, l in enumerate(LABEL_ORDER)}

if res is not None:
    cm_raw = np.zeros((3, 3), dtype=int)
    cm_wt = np.zeros((3, 3))
    for r in res.itertuples():
        i, j = lab_ix[r.true_label], lab_ix[r.predicted]
        cm_raw[i, j] += 1
        cm_wt[i, j] += r.weight

    def with_row_pct(cm):
        out = pd.DataFrame(index=LABEL_ORDER, columns=LABEL_ORDER)
        for i in range(3):
            row_sum = cm[i].sum()
            for j in range(3):
                pct = 100 * cm[i, j] / row_sum if row_sum else 0.0
                out.iloc[i, j] = f"{int(round(cm[i, j])):>6,} ({pct:5.1f}%)"
        return out

    print("RAW confusion matrix (rows = truth, cols = predicted) - Table 2:")
    print(with_row_pct(cm_raw.astype(float)).to_string())
    print("\nWEIGHTED (population-scale) confusion matrix:")
    print(with_row_pct(cm_wt).to_string())

    prec = np.divide(np.diag(cm_wt), cm_wt.sum(axis=0),
                     out=np.zeros(3), where=cm_wt.sum(axis=0) > 0)
    rec = np.divide(np.diag(cm_wt), cm_wt.sum(axis=1),
                    out=np.zeros(3), where=cm_wt.sum(axis=1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec,
                   out=np.zeros(3), where=(prec + rec) > 0)
    print("\nPer-class metrics on the weighted matrix:")
    for i, l in enumerate(LABEL_ORDER):
        print(f"  {l:16s} precision {prec[i]:.4f}  recall {rec[i]:.4f}  F1 {f1[i]:.4f}")
    print(f"\nPopulation-weighted macro F1: {f1.mean():.4f}   (paper: 0.9332)")
    print(f"Eruption-only macro F1:       {f1[:2].mean():.4f}   (paper: 0.8998)")
else:
    print("Skipped - no review outcomes loaded.")


## 7. Threshold sweeps on the reviewed data

Raising a class's threshold demotes weaker detections to No Eruption. The
sweep shows, per class, the TP/FP trade and the threshold at which false
positives vanish entirely (paper: precision 1.00 for Large at 0.63 keeping
46 of 48 true positives; for Small at 0.61 keeping 311 of 406).

Caveats, unchanged from the research notebook: thresholds can only be
*raised* post hoc (an event that never fired was never reviewed), event
clustering stays fixed at the original thresholds, and because the optimum
is chosen on the same reviewed data it is an in-sample estimate: validate
on another month before operational use.

In [ ]:
if res is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    for ax, pred_class, t0 in [(axes[0], "Large Eruption", config.THRESH_LARGE),
                               (axes[1], "Small Eruption", config.THRESH_SMALL)]:
        sub = res[res["predicted"] == pred_class]
        probs = sub["max_prob"].to_numpy()
        is_tp = (sub["true_label"] == pred_class).to_numpy()
        grid = np.arange(t0, 1.0005, 0.001)
        tp = np.array([(probs > t)[is_tp].sum() for t in grid])
        fp = np.array([(probs > t)[~is_tp].sum() for t in grid])
        ax.plot(grid, tp, color="#1e7d1e", lw=2.2, label="True positives")
        ax.plot(grid, fp, color="#c0392b", lw=2.2, label="False positives")
        hits = np.flatnonzero(fp == 0)
        if len(hits):
            t_fp0 = grid[hits[0]]
            ax.axvline(t_fp0, color="black", ls="--", alpha=0.6,
                       label=f"FP = 0 at t = {t_fp0:.3f} (TP kept: {tp[hits[0]]})")
            print(f"{pred_class}: FP reaches 0 at t = {t_fp0:.3f}, keeping "
                  f"{tp[hits[0]]} of {int(is_tp.sum())} true positives")
        ax.axvline(t0, color="gray", ls=":", label=f"operating t = {t0:.4f}")
        ax.set_xlabel("Threshold"); ax.set_ylabel("Event count")
        ax.set_title(f"{pred_class} (n reviewed = {len(sub)})"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(config.OUTPUT_DIR / "tp_fp_sweep.png", dpi=300); plt.show()
else:
    print("Skipped - no review outcomes loaded.")


## 8. Missed small eruptions: near-threshold or deep misses?

A miss is a reviewed No-Eruption item the reviewer reclassified as Small
Eruption. Comparing each miss's smoothed P(small) against the threshold
separates edge cases from genuine blind spots;
the weighted totals estimate how many small eruptions the whole month's
run missed (the paper's ~30% missed-small figure comes from this analysis
combined with the recall estimate).

In [ ]:
if res is not None:
    EDGE_MARGIN = 0.10
    misses = res[(res["predicted"] == "No Eruption")
                 & (res["true_label"] == "Small Eruption")]
    if len(misses):
        idx = misses["center_index"].astype(int).to_numpy()
        file_times = pd.to_datetime(misses["center_time"]).to_numpy()
        pipe_times = df.iloc[idx]["Timestamp"].to_numpy()
        assert (np.abs(pipe_times - file_times) < np.timedelta64(1, "ms")).all(), \
            "center_index/timestamp mismatch between review file and pipeline"

        p = df.iloc[idx]["Small Eruption"].to_numpy()
        w = misses["weight"].to_numpy()
        edge = p >= config.THRESH_SMALL - EDGE_MARGIN
        print(f"Reviewed misses: {len(misses)}  ->  weighted estimate "
              f"{w.sum():,.0f} missed small eruptions this month")
        print(f"Near-threshold (within {EDGE_MARGIN} of the {config.THRESH_SMALL} "
              f"threshold): raw {int(edge.sum())}, weighted {w[edge].sum():,.0f}")
        print(f"Deep misses: raw {int((~edge).sum())}, weighted {w[~edge].sum():,.0f}")
        est_recall = 406 / (406 + w.sum())
        print(f"\nImplied population recall for Small Eruption: "
              f"406 / (406 + {w.sum():,.0f}) = {est_recall:.4f}")
    else:
        print("No reviewed misses found.")
else:
    print("Skipped - no review outcomes loaded.")


## 9. Weekly probability panels (Figure 6)

Per-frame smoothed probabilities for the month: blue = Small, red = Large,
dark gray = camera offline, light gray band = below-threshold zone. The
first week's dense blue activity is the period of increased small eruptions
discussed in the paper.

In [ ]:
start_midnight = df["Timestamp"].min().normalize()
end_time = df["Timestamp"].max()
week = pd.Timedelta(days=7)
weeks = pd.date_range(start=start_midnight, end=end_time, freq="7D")
if weeks[-1] < end_time:
    weeks = weeks.union(pd.DatetimeIndex([end_time]))
n_weeks = len(weeks) - 1

fig, axes = plt.subplots(n_weeks, 1, figsize=(22, 4.5 * n_weeks))
axes = np.atleast_1d(axes)
for i in range(n_weeks):
    ax = axes[i]
    w_start, w_end = weeks[i], weeks[i + 1]
    sub = df[(df["Timestamp"] >= w_start) & (df["Timestamp"] < w_end)]
    if sub.empty:
        ax.set_visible(False)
        continue
    if i == 0 and sub["Timestamp"].iloc[0] > w_start:
        ax.axvspan(w_start, sub["Timestamp"].iloc[0], color="#505050", zorder=5)
    for idx in sub.index[sub["delta"] > pd.Timedelta(seconds=30)]:
        if idx - 1 in sub.index:
            ax.axvspan(sub.loc[idx - 1, "Timestamp"], sub.loc[idx, "Timestamp"],
                       color="#505050", zorder=5)
    ax.axhspan(0, config.THRESH_LARGE, color="gray", alpha=0.5, zorder=4)
    ax.fill_between(sub["Timestamp"], sub["Small Eruption"], 0, color="royalblue", zorder=2)
    ax.fill_between(sub["Timestamp"], sub["Large Eruption"], 0, color="firebrick", zorder=3)
    ax.set_xlim(w_start, w_start + week); ax.set_ylim(0, 1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.text(0.005, 0.95, f"({chr(ord('a') + i)})", transform=ax.transAxes,
            fontsize=14, fontweight="bold", va="top")

axes[-1].legend(handles=[mpatches.Patch(color="royalblue", label="Small Eruption"),
                         mpatches.Patch(color="firebrick", label="Large Eruption"),
                         mpatches.Patch(color="#505050", label="Camera offline"),
                         mpatches.Patch(color="gray", alpha=0.5, label="Below threshold")],
                loc="lower right")
fig.text(0.06, 0.5, "Probability", va="center", rotation="vertical", fontsize=16)
fig.suptitle("StromNet probability classifications: December 2015", fontsize=18)
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / "figure6_weekly_probabilities.png", dpi=200)
plt.show()


---
## 10. Interactive manual review

Only needed to redo (or extend) the review; the published outcomes already
ship in `data/review_results_2015_12.csv`.

In [ ]:
# Review order. Labels are keyed by a stable id rather than by position, so
# any ordering here is safe and a resumed session reconciles regardless.
#   "full"            - shuffle every item together, so the three categories
#                       interleave. Removes both within-category drift and any
#                       anchoring that comes from knowing which category block
#                       you are currently in.
#   "within_category" - shuffle inside each category, categories still run in
#                       sequence. This is what the published review used.
#   "none"            - chronological, as built.
SHUFFLE_MODE = "full"
RUN_INTERACTIVE_REVIEW = True   # set True to launch the review UI

if SHUFFLE_MODE == "full":
    rng = random.Random(config.RANDOM_SEED)
    rng.shuffle(review_items)
elif SHUFFLE_MODE == "within_category":
    rng = random.Random(config.RANDOM_SEED)
    by_cat = {}
    for r in review_items:
        by_cat.setdefault(r["category"], []).append(r)
    ordered = []
    for cat in LABEL_ORDER:
        rng.shuffle(by_cat.get(cat, []))
        ordered.extend(by_cat.get(cat, []))
    review_items = ordered
elif SHUFFLE_MODE != "none":
    raise ValueError("SHUFFLE_MODE must be 'full', 'within_category', or 'none'.")

switches = sum(1 for a, b in zip(review_items, review_items[1:])
               if a["category"] != b["category"])
print(f"Review queue ordering: {SHUFFLE_MODE} (seed {config.RANDOM_SEED}); "
      f"{len(review_items):,} items, {switches:,} category changes.")

labels = {}

if RUN_INTERACTIVE_REVIEW:
    import io
    import ipywidgets as widgets
    from IPython.display import display
    from PIL import Image, ImageDraw, ImageFont

    PROGRESS_PKL = config.OUTPUT_DIR / "review_progress_2015_12.pkl"
    CLASSES = LABEL_ORDER
    if PROGRESS_PKL.exists():
        with open(PROGRESS_PKL, "rb") as f:
            labels.update(pickle.load(f))
        print(f"Resumed {len(labels)} prior ratings.")

    queue = review_items

    try:
        FONT = ImageFont.load_default()
    except Exception:
        FONT = None

    def composite(item, h=180, gap=8):
        frames = []
        for p in item["image_paths"]:
            try:
                img = Image.open(p).convert("RGB")
                frames.append(img.resize((int(img.width * h / img.height), h)))
            except Exception:
                frames.append(Image.new("RGB", (240, h), (20, 20, 20)))
        canvas = Image.new("RGB", (sum(f.width for f in frames) + gap * 4, h + 22), (245, 245, 245))
        draw = ImageDraw.Draw(canvas)
        x = 0
        for f_img, ix in zip(frames, item["window_idxs"]):
            draw.text((x + 2, 2), df.iloc[ix]["Timestamp_Str"], fill=(20, 20, 20), font=FONT)
            canvas.paste(f_img, (x, 22))
            x += f_img.width + gap
        buf = io.BytesIO(); canvas.save(buf, format="PNG")
        return buf.getvalue()

    state = {"pos": next((i for i, r in enumerate(queue) if r["id"] not in labels), 0)}
    header = widgets.HTML()
    strip = widgets.Image(format="png")
    radio = widgets.RadioButtons(options=CLASSES, description="True class:")
    nxt = widgets.Button(description="Commit & Next", button_style="primary")
    prev = widgets.Button(description="Previous")
    prog = widgets.HTML()

    def refresh():
        item = queue[state["pos"]]
        header.value = (f"<b>Item {state['pos'] + 1}/{len(queue)}</b> - predicted "
                        f"<b>{item['category']}</b> (p={item['max_prob']:.3f}) - {item['center_time']}")
        radio.value = labels.get(item["id"], item["category"])
        prog.value = f"Labelled: <b>{len(labels)}/{len(queue)}</b>"
        strip.value = composite(item)

    def on_next(_):
        item = queue[state["pos"]]
        labels[item["id"]] = radio.value
        with open(PROGRESS_PKL, "wb") as f:
            pickle.dump(labels, f)
        state["pos"] = min(len(queue) - 1, state["pos"] + 1)
        refresh()

    def on_prev(_):
        state["pos"] = max(0, state["pos"] - 1)
        refresh()

    nxt.on_click(on_next); prev.on_click(on_prev)
    display(header, strip, radio, widgets.HBox([prev, nxt]), prog)
    refresh()
else:
    print("Interactive review disabled (RUN_INTERACTIVE_REVIEW = False).")

In [ ]:
# --- Export reviewed outcomes ------------------------------------------------
# Run after finishing (or pausing) an interactive review session.  Writes to
# outputs/ .
if labels:
    rows = [dict(id=r["id"], predicted=r["category"], true_label=labels[r["id"]],
                 correct=labels[r["id"]] == r["category"], stratum=r["stratum"],
                 weight=r["weight"], center_time=r["center_time"],
                 center_index=r["center_index"], max_prob=r["max_prob"])
            for r in review_items if r["id"] in labels]
    out = config.OUTPUT_DIR / "review_results_2015_12_new.csv"
    pd.DataFrame(rows).to_csv(out, index=False)
    print(f"Wrote {len(rows)} rows -> {out}")
else:
    print("Nothing to export yet (no ratings recorded in this session).")


---
## 11. Session metrics

Scores the ratings made in the interactive review above, exactly as the
metrics in section 6 score the shipped published review: a raw confusion
matrix of what was actually clicked, a Horvitz-Thompson weighted matrix
estimating the full month, and precision/recall/F1 in both views.

In [ ]:
scored = [r for r in review_items if r["id"] in labels]

if not scored:
    print("No session ratings to score (run the interactive review above, or "
          "see section 6 for the shipped published review).")
else:
    print(f"Items scored: {len(scored)} / {len(review_items)} "
          f"({100 * len(scored) / len(review_items):.1f}% of the queue)")
    strata = {}
    for r in review_items:
        strata.setdefault(r["stratum"], [0, 0])
        strata[r["stratum"]][1] += 1
        if r["id"] in labels:
            strata[r["stratum"]][0] += 1
    for name, (done, total) in strata.items():
        print(f"  {name:16s} {done:>5,} / {total:,}")
    if len(scored) < len(review_items):
        print("Partial review: weighted estimates are unbiased only once each "
              "stratum is fully (or randomly) covered.")
    stale = len(labels) - len(scored)
    if stale:
        print(f"({stale} saved rating(s) match no current queue item and are ignored.)")

    cm_raw = np.zeros((3, 3), dtype=int)
    cm_weighted = np.zeros((3, 3))
    for r in scored:
        i, j = lab_ix[labels[r["id"]]], lab_ix[r["category"]]
        cm_raw[i, j] += 1
        cm_weighted[i, j] += r.get("weight", 1.0)

    def metrics_from_cm(cm):
        prec = np.divide(np.diag(cm), cm.sum(axis=0),
                         out=np.zeros(3), where=cm.sum(axis=0) > 0)
        rec = np.divide(np.diag(cm), cm.sum(axis=1),
                        out=np.zeros(3), where=cm.sum(axis=1) > 0)
        f1 = np.divide(2 * prec * rec, prec + rec,
                       out=np.zeros(3), where=(prec + rec) > 0)
        support = cm.sum(axis=1)
        weighted = (f1 * support).sum() / support.sum() if support.sum() else 0.0
        return prec, rec, f1, f1.mean(), weighted

    p_raw, r_raw, f1_raw, macro_raw, wt_raw = metrics_from_cm(cm_raw.astype(float))
    p_wt, r_wt, f1_wt, macro_wt, wt_wt = metrics_from_cm(cm_weighted)

    print("\nRAW confusion matrix (review-set counts, rows = truth):")
    print(pd.DataFrame(cm_raw, index=LABEL_ORDER, columns=LABEL_ORDER).to_string())
    print("\nWEIGHTED confusion matrix (population-scale estimate):")
    print(pd.DataFrame(np.round(cm_weighted).astype(int),
                       index=LABEL_ORDER, columns=LABEL_ORDER).to_string())

    print(f"\n{'':17s}   {'RAW (review set)':^25s}   {'WEIGHTED (population)':^25s}")
    print(f"{'':17s}   {'P':>7s} {'R':>7s} {'F1':>7s}   {'P':>7s} {'R':>7s} {'F1':>7s}")
    for i, lbl in enumerate(LABEL_ORDER):
        print(f"{lbl:17s}   {p_raw[i]:>7.4f} {r_raw[i]:>7.4f} {f1_raw[i]:>7.4f}   "
              f"{p_wt[i]:>7.4f} {r_wt[i]:>7.4f} {f1_wt[i]:>7.4f}")
    print(f"\n{'Macro F1':17s}   {macro_raw:>23.4f}   {macro_wt:>25.4f}")
    print(f"{'Support-wtd F1':17s}   {wt_raw:>23.4f}   {wt_wt:>25.4f}")
    print(f"{'Eruption-only F1':17s}   {f1_raw[:2].mean():>23.4f}   {f1_wt[:2].mean():>25.4f}")
    print("(eruption-only = mean of Large + Small F1, excluding the dominant "
          "background class)")

In [ ]:
if scored:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, cm, title in [(axes[0], cm_raw.astype(float), "Raw (review set)"),
                          (axes[1], cm_weighted, "Weighted (population estimate)")]:
        ax.imshow(np.log1p(cm), cmap="Blues")
        ax.set_xticks(range(3), LABEL_ORDER, rotation=20, ha="right")
        ax.set_yticks(range(3), LABEL_ORDER)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"{title}  (color: log scale)")
        for i in range(3):
            for j in range(3):
                v = cm[i, j]
                ax.text(j, i, f"{v:,.0f}", ha="center", va="center",
                        color="white" if np.log1p(v) > 0.6 * np.log1p(cm).max() else "black")
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / "session_review_confusion.png", dpi=300)
    plt.show()
else:
    print("Nothing to plot (no session ratings).")